# Custom Data RAG Notebook

Bu notebook hocanın verdiği custom doküman koleksiyonunu aynı RAG sisteminde çalıştırmak için hazırlanmıştır.

Desteklenen inputlar:

```text
custom_documents.json
custom_documents.jsonl
custom_documents.csv
custom_docs/ klasörü içindeki .txt, .md, opsiyonel .pdf dosyaları
```

Önerilen JSON formatı:

```json
[
  {
    "id": "doc_001",
    "title": "Belge başlığı",
    "text": "Belge metni...",
    "source_url": "optional"
  }
]
```

Hukuk maddesi gibi metadata varsa şu alanlar da kullanılabilir:

```text
law_name, law_no, article_no, article_title, text, source_url
```

In [ ]:
!nvidia-smi

## 1. Paketler ve Ortam

In [ ]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import importlib.util
import subprocess
import sys

required = {
    "sentence_transformers": "sentence-transformers",
    "faiss": "faiss-cpu",
    "transformers": "transformers",
    "peft": "peft",
    "bitsandbytes": "bitsandbytes",
    "accelerate": "accelerate",
    "pypdf": "pypdf",
}
missing = [pip_name for module_name, pip_name in required.items() if importlib.util.find_spec(module_name) is None]
print("Missing packages:", missing)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

## 2. Scriptleri Kopyala

In [ ]:
from pathlib import Path
import shutil

INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working/legal-rag-custom")

for path in [
    WORK_DIR / "scripts",
    WORK_DIR / "data/processed",
    WORK_DIR / "data/index",
    WORK_DIR / "data/eval",
]:
    path.mkdir(parents=True, exist_ok=True)

def find_input_file(name: str) -> Path:
    matches = sorted(INPUT_ROOT.rglob(name))
    if not matches:
        raise FileNotFoundError(f"{name} not found under {INPUT_ROOT}. Kaggle dataset'e ekledin mi?")
    return matches[0]

def copy_required(name: str, dest_dir: Path) -> Path:
    src = find_input_file(name)
    dst = dest_dir / name
    shutil.copy2(src, dst)
    print(f"Copied {src} -> {dst}")
    return dst

for name in [
    "prepare_custom_corpus.py",
    "build_faiss_index.py",
    "evaluate_retrieval.py",
    "rag_answer.py",
]:
    copy_required(name, WORK_DIR / "scripts")

print("Scripts copied.")

## 3. Custom Doküman Koleksiyonunu Bul

In [ ]:
candidates = []
for name in [
    "custom_documents.json",
    "custom_documents.jsonl",
    "custom_documents.csv",
]:
    candidates.extend(INPUT_ROOT.rglob(name))

candidates.extend([p for p in INPUT_ROOT.rglob("custom_docs") if p.is_dir()])

print("Custom data candidates:")
for path in candidates:
    print("-", path)

if not candidates:
    raise FileNotFoundError(
        "Custom doküman bulunamadı. Kaggle dataset'e custom_documents.json/jsonl/csv "
        "veya custom_docs klasörü ekle."
    )

CUSTOM_INPUT = candidates[0]
print("Using custom input:", CUSTOM_INPUT)

## 4. Custom Corpus ve Chunks Üret

In [ ]:
import subprocess

CUSTOM_CORPUS = WORK_DIR / "data/processed/retrieval_corpus_custom.json"
CUSTOM_CHUNKS = WORK_DIR / "data/processed/retrieval_chunks_custom.json"

cmd = [
    "python", "-u", str(WORK_DIR / "scripts/prepare_custom_corpus.py"),
    "--input", str(CUSTOM_INPUT),
    "--corpus-out", str(CUSTOM_CORPUS),
    "--chunks-out", str(CUSTOM_CHUNKS),
    "--collection-name", "Custom Documents",
    "--max-words", "450",
    "--overlap-words", "64",
]
subprocess.run(cmd, check=True)

## 5. Custom FAISS Index Kur

In [ ]:
CUSTOM_INDEX = WORK_DIR / "data/index/faiss_custom_bge_m3.index"
CUSTOM_METADATA = WORK_DIR / "data/index/metadata_custom_bge_m3.json"
CUSTOM_CONFIG = WORK_DIR / "data/index/index_config_custom_bge_m3.json"

cmd = [
    "python", "-u", str(WORK_DIR / "scripts/build_faiss_index.py"),
    "--chunks", str(CUSTOM_CHUNKS),
    "--index-out", str(CUSTOM_INDEX),
    "--metadata-out", str(CUSTOM_METADATA),
    "--config-out", str(CUSTOM_CONFIG),
    "--model", "BAAI/bge-m3",
    "--device", "cuda",
    "--batch-size", "8",
]
subprocess.run(cmd, check=True)

## 6. LoRA Adapter Varsa RAG Demo Yükle

In [ ]:
adapter_configs = sorted(INPUT_ROOT.rglob("adapter_config.json"))
ADAPTER_PATH = adapter_configs[0].parent if adapter_configs else None
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"

print("Adapter:", ADAPTER_PATH)

import sys
sys.path.insert(0, str(WORK_DIR / "scripts"))
from rag_answer import LegalRAG

rag = LegalRAG(
    index_path=CUSTOM_INDEX,
    metadata_path=CUSTOM_METADATA,
    config_path=CUSTOM_CONFIG,
    corpus_path=CUSTOM_CORPUS,
    alpha=0.70,
    dense_candidates=300,
    bm25_candidates=100,
    preliminary_top_k=50,
)
rag.load_retriever(embedding_device="cpu")

if ADAPTER_PATH:
    rag.load_llm(BASE_MODEL, adapter_path=ADAPTER_PATH, load_in_4bit=True)
else:
    print("LoRA adapter yok. Sadece retrieval demo çalıştırılabilir.")

## 7. Custom Retrieval / RAG Testi

In [ ]:
question = "Bu dokümanlara göre en önemli düzenleme nedir?"

contexts = rag.retrieve(question, top_k=5)
print("Retrieved contexts:")
for ctx in contexts:
    print(f"[{ctx['rank']}] {ctx['citation']} | {ctx['parent_id']} | score={ctx['score']:.4f}")

if ADAPTER_PATH:
    result = rag.answer(question, top_k=5, max_new_tokens=384, do_sample=False)
    print("\nAnswer:")
    print(result["answer"])

## 8. Custom Gold Benchmark Varsa Değerlendir

In [ ]:
benchmark_candidates = []
for name in ["custom_benchmark.json", "custom_benchmark.csv"]:
    benchmark_candidates.extend(INPUT_ROOT.rglob(name))

if benchmark_candidates:
    CUSTOM_BENCHMARK = benchmark_candidates[0]
    print("Using custom benchmark:", CUSTOM_BENCHMARK)
    cmd = [
        "python", "-u", str(WORK_DIR / "scripts/evaluate_retrieval.py"),
        "--benchmark", str(CUSTOM_BENCHMARK),
        "--corpus", str(CUSTOM_CORPUS),
        "--chunks", str(CUSTOM_CHUNKS),
        "--index", str(CUSTOM_INDEX),
        "--metadata", str(CUSTOM_METADATA),
        "--config", str(CUSTOM_CONFIG),
        "--mode", "hybrid",
        "--embedding-device", "cuda",
        "--top-k", "10",
        "--alpha", "0.70",
        "--dense-candidates", "300",
        "--bm25-candidates", "100",
        "--preliminary-top-k", "50",
        "--output", str(WORK_DIR / "data/eval/custom_retrieval_eval.json"),
    ]
    subprocess.run(cmd, check=True)
else:
    print("No custom_benchmark.json/csv found. Skipping evaluation.")